# Huber GLM Slope-Core Submission

## 1. Setup

In [1]:
"Imports, paths, constants, and data discovery."

from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

KAGGLE_ROOT = Path("/kaggle")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")

MODEL_NAME = "huber_glm_slope_core"
MAX_TARGET_ROWS_PER_WELL = 140
HUBER_ALPHA = 0.0001
HUBER_EPSILON = 1.35
HUBER_MAX_ITER = 1000

FEATURE_COLUMNS = [
    "distance_from_boundary_md",
    "eval_progress",
    "md_norm",
    "dZ_from_boundary",
    "horizontal_dist_from_boundary",
    "tvt_slope_last_10",
    "tvt_slope_last_25",
    "tvt_slope_last_50",
    "tvt_slope_last_100",
    "z_slope_last_25",
    "z_slope_last_50",
    "z_slope_last_100",
    "tvt_slope_last_10_projected_delta",
    "tvt_slope_last_25_projected_delta",
    "tvt_slope_last_50_projected_delta",
    "tvt_slope_last_100_projected_delta",
    "last_known_tvt",
]


def local_data_candidates():
    candidates = [Path.cwd() / "data" / "raw"]
    candidates.extend(parent / "data" / "raw" for parent in Path.cwd().parents)
    return candidates


def resolve_base_dir():
    known = Path("/kaggle/input/rogii-wellbore-geology-prediction")
    for candidate in [known] + local_data_candidates():
        if (candidate / "sample_submission.csv").exists() and (candidate / "train").exists() and (candidate / "test").exists():
            return candidate
    if KAGGLE_INPUT_ROOT.exists():
        for sample_path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv"):
            candidate = sample_path.parent
            if (candidate / "train").exists() and (candidate / "test").exists():
                return candidate
        children = sorted(str(path) for path in KAGGLE_INPUT_ROOT.glob("*"))
        raise FileNotFoundError(f"Could not find competition data under /kaggle/input. Children: {children}")
    raise FileNotFoundError("Could not find Kaggle or local competition data.")


BASE_DIR = resolve_base_dir()

if KAGGLE_ROOT.exists():
    OUTPUT_PATH = KAGGLE_OUTPUT / "submission.csv"
else:
    LOCAL_ROOT = BASE_DIR.parent.parent
    OUTPUT_PATH = LOCAL_ROOT / "artifacts" / "reports" / f"submission__glm_huber__{date.today().isoformat()}.csv"

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
SAMPLE_PATH = BASE_DIR / "sample_submission.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("n_features:", len(FEATURE_COLUMNS))

BASE_DIR: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw
OUTPUT_PATH: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\submission__glm_huber__2026-06-22.csv
n_features: 17


## 2. Helpers

In [2]:
"Loaders, slope-core feature construction, and deterministic sampling."


def discover_wells(directory):
    return sorted(path.name.replace("__horizontal_well.csv", "") for path in Path(directory).glob("*__horizontal_well.csv"))


def load_horizontal_well(path):
    df = pd.read_csv(path)
    df.index.name = "row_idx"
    df["row_idx"] = df.index
    return df


def load_sample_submission(path):
    sample = pd.read_csv(path)
    split = sample["id"].str.rsplit("_", n=1, expand=True)
    sample["wellname"] = split[0]
    sample["row_idx"] = split[1].astype(int)
    return sample


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, row_indices=None, max_train_rows=None, require_target=False):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    known_mask = horizontal["TVT_input"].notna()
    if require_target:
        candidate_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    else:
        candidate_mask = horizontal["TVT_input"].isna()
    if row_indices is not None:
        row_index_set = set(int(idx) for idx in row_indices)
        candidate_mask = candidate_mask & horizontal["row_idx"].isin(row_index_set)
    if int(candidate_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}

    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    if max_train_rows is None:
        target_positions = np.flatnonzero(candidate_mask.to_numpy())
    else:
        target_positions = deterministic_sample_positions(candidate_mask, max_train_rows)
    all_target_positions = np.flatnonzero(candidate_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
        }
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        if require_target:
            row["TVT"] = float(horizontal["TVT"].iloc[pos])
            row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        rows.append(row)
    return pd.DataFrame(rows)


def make_huber_model():
    return Pipeline([
        (
            "features",
            ColumnTransformer([
                (
                    "numeric",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]),
                    FEATURE_COLUMNS,
                )
            ], remainder="drop", sparse_threshold=0.0),
        ),
        ("huber", HuberRegressor(alpha=HUBER_ALPHA, epsilon=HUBER_EPSILON, max_iter=HUBER_MAX_ITER)),
    ])

## 3. Train Huber GLM

In [3]:
"Build sampled target-zone training rows and fit robust signed-delta Huber GLM."

train_frames = []
train_wells = discover_wells(TRAIN_DIR)
for well in train_wells:
    frame = build_slope_core_features_for_well(
        well,
        TRAIN_DIR,
        max_train_rows=MAX_TARGET_ROWS_PER_WELL,
        require_target=True,
    )
    if not frame.empty:
        train_frames.append(frame)

train_df = pd.concat(train_frames, ignore_index=True)
model = make_huber_model()
model.fit(train_df[FEATURE_COLUMNS], train_df["delta_tvt_from_boundary"].to_numpy(float))

print("train_wells:", len(train_wells))
print("train_rows:", len(train_df))
print("model:", MODEL_NAME)
print("config:", {"alpha": HUBER_ALPHA, "epsilon": HUBER_EPSILON, "max_iter": HUBER_MAX_ITER})

train_wells: 773
train_rows: 108220
model: huber_glm_slope_core
config: {'alpha': 0.0001, 'epsilon': 1.35, 'max_iter': 1000}


## 4. Write Submission

In [4]:
"Generate predictions in sample_submission order and write the CSV."

sample = load_sample_submission(SAMPLE_PATH)
predictions = {}

for well, group in sample.groupby("wellname", sort=True):
    frame = build_slope_core_features_for_well(
        well,
        TEST_DIR,
        row_indices=group["row_idx"].tolist(),
        max_train_rows=None,
        require_target=False,
    )
    if frame.empty:
        continue
    pred_delta = model.predict(frame[FEATURE_COLUMNS])
    pred_tvt = frame["last_known_tvt"].to_numpy(float) + pred_delta
    for row_idx, value in zip(frame["row_idx"].astype(int), pred_tvt):
        predictions[f"{well}_{row_idx}"] = float(value)

submission = sample[["id"]].copy()
submission["tvt"] = submission["id"].map(predictions)

assert list(submission.columns) == ["id", "tvt"]
assert len(submission) == len(sample)
assert submission["id"].equals(sample["id"])
assert submission["tvt"].notna().all()
assert np.isfinite(submission["tvt"].to_numpy()).all()

submission.to_csv(OUTPUT_PATH, index=False)

print("submission_rows:", len(submission))
print("submission_path:", OUTPUT_PATH)
submission.head()

submission_rows: 14151
submission_path: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\submission__glm_huber__2026-06-22.csv


,id,tvt
0,000d7d20_1442,11744.928078
1,000d7d20_1443,11744.928130
2,000d7d20_1444,11744.928183
3,000d7d20_1445,11744.928237
4,000d7d20_1446,11744.928289
